# Libraries and global variables

In [1]:
import sys
import time
from pathlib import Path
import json

import numpy as np
import pandas as pd

import torch
from fastai.vision.all import (
    DataBlock, ImageBlock, CategoryBlock,
    ColReader, ColSplitter,
    vision_learner,
    SaveModelCallback, EarlyStoppingCallback, CSVLogger,
    accuracy, RocAuc, F1Score, ClassificationInterpretation,
    Resize, DataLoaders
)
from fastai.callback.fp16 import MixedPrecision
import timm

from fastai.vision.augment import (
    Brightness, Contrast, aug_transforms
)

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
!cp -r "/content/drive/MyDrive/Work/10 Foodbegood/project_gn_food_estimator_v1" "/content/project_gn_food_estimator_v1"

## Directory layout & paths

In [4]:
PLATFORM = "colab" # "colab", "kaggle", "local"

if PLATFORM == "colab":
    ROOT = Path("/content/project_gn_food_estimator_v1")
elif PLATFORM == "kaggle":
    ROOT = Path("/kaggle/working/project_gn_food_estimator")
else:
    ROOT = Path(".").resolve()

In [5]:
DATA_DIR      = ROOT / "data"
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EXPORTS_DIR   = DATA_DIR / "exports"
MODELS_DIR    = ROOT / "models"
TRAINING_DIR  = ROOT / "training"
LOGS_DIR      = TRAINING_DIR / "logs"

# Labels file
LS_MODEL2_JSON = EXPORTS_DIR / "model2_fill_level.json"

# Model checkpoint path
MODEL2_PATH = MODELS_DIR / "model2_fill.pth"

log_path = LOGS_DIR / "model2_training_log.csv"

In [6]:
def setup_directories():
    """
    Ensure output directories exist.
    """
    MODELS_DIR.mkdir(parents = True, exist_ok = True)
    LOGS_DIR.mkdir(parents = True, exist_ok = True)
    print(f"[Setup] Models dir: {MODELS_DIR}")
    print(f"[Setup] Logs dir: {LOGS_DIR}")

In [7]:
setup_directories()

[Setup] Models dir: /content/project_gn_food_estimator_v1/models
[Setup] Logs dir: /content/project_gn_food_estimator_v1/training/logs


## Parameters

In [8]:
# Dataset split ratios
SPLIT_TRAIN = 0.80
SPLIT_VAL   = 0.10
SPLIT_TEST  = 0.10
RANDOM_SEED = 42

In [9]:
# Class labels — must match Label Studio annotation labels exactly
MODEL2_CLASSES = ['empty', 'full', 'high', 'low', 'medium']

In [10]:
# If classifier confidence falls below this threshold, ask user to retake photo
CONFIDENCE_THRESHOLD = 0.70

In [11]:
# Image pre-processing
IMAGE_SIZE = 512
NORM_MEAN = [0.485, 0.456, 0.406] # ImageNet statistics
NORM_STD = [0.229, 0.224, 0.225]
HIST_EQ_ENABLED = True

In [12]:
# Data augmentation toggles
AUG_BRIGHTNESS = True
AUG_ROTATION   = True
AUG_BLUR       = True
AUG_SCALING    = True
AUG_FLIPPING   = True
AUG_CONTRAST   = True

## Training hyperparameters

In [13]:
# Model #2 — EfficientNet-B0, image classification
MODEL2_BACKBONE        = "efficientnet_b0"
MODEL2_BATCH_SIZE      = 16      # Lighter model, larger batch fine on T4
MODEL2_EPOCHS_FROZEN   = 5
MODEL2_EPOCHS_UNFROZEN = 10
MODEL2_LR_FROZEN       = 1e-3
MODEL2_LR_UNFROZEN     = 1e-4
MODEL2_MIXED_PREC      = True

In [14]:
def check_gpu():
    """
    Report GPU availability and VRAM.
    """
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

        print(f"[GPU] {gpu_name} - {vram_gb:.1f} GB VRAM")

        if vram_gb < 8.0:
            print(
                f"[GPU] WARNING: Less than 8 GB VRAM detected. "
                f"Consider reducing MODEL2_BATCH_SIZE (currently {MODEL2_BATCH_SIZE})."
            )
    else:
        print("[GPU] No CUDA device found - training will run on CPU.")
    return "cuda" if torch.cuda.is_available() else "cpu"

In [15]:
DEVICE = check_gpu()

[GPU] Tesla T4 - 15.6 GB VRAM


# Data Preprocessing

## Data augmentation pipeline

In [16]:
def get_classification_transforms(size: int = IMAGE_SIZE):
    """
    fastai's aug_transforms() covers: flip, rotation, zoom (scaling),
    warp, lighting (brightness + contrast), and blur.

    Returns
        tuple[list, list]
            (train_transforms, valid_transforms)
    """
    aug_kwargs = dict(
        size = size,

        # Flipping: only horizontal — vertical flip is disabled for top-down food
        do_flip = AUG_FLIPPING,
        flip_vert = False,

        # Rotation: ±20° captures realistic hand-angle variation
        max_rotate = 20.0 if AUG_ROTATION else 0.0,

        # Scaling / zoom: 1.0–1.15 — mild zoom to simulate distance variation
        min_zoom = 1.0,
        max_zoom = 1.15 if AUG_SCALING else 1.0,

        # Lighting (brightness + contrast handled together by aug_transforms)
        max_lighting = 0.3 if (AUG_BRIGHTNESS or AUG_CONTRAST) else 0.0,

        # Blur: p = 0.3 means applied to ~30% of batches
        #max_blur = 2.0 if AUG_BLUR else 0.0,

        # Warp: small perspective warp simulates slight camera tilt
        max_warp = 0.1,

        p_affine = 0.75,   # probability of applying affine transforms
        p_lighting = 0.75  # probability of applying lighting transforms
    )

    train_tfms = aug_transforms(**aug_kwargs)
    valid_tfms = []  # No augmentation on validation / test sets

    return train_tfms, valid_tfms

In [17]:
train_tfms, _ = get_classification_transforms(size = IMAGE_SIZE)

# Loading data

In [18]:
def _load_ls_json(json_path: Path) -> list:
    """
    Load and minimally validate a Label Studio native JSON export.

    Returns
    -------
    list of task dicts
    """
    if not json_path.exists():
        raise FileNotFoundError(
            f"Label Studio JSON not found: {json_path}\n"
            f"Export your project using: Project → Export → JSON"
        )
    with open(json_path, "r") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(
            f"Expected a JSON list (Label Studio native format) but got "
            f"{type(data).__name__}. "
            f"If you exported as 'COCO JSON', re-export using plain 'JSON'."
        )
    if len(data) == 0:
        raise ValueError("JSON file is empty — no tasks found.")
    print(f"  Loaded {len(data)} tasks from {json_path.name}")

    return data


def _extract_label(task: dict) -> str | None:
    """
    Extract the fill-level label from a single Label Studio task object.

    Navigates: annotations[0] → result[0] → value → choices[0]

    Returns None if the annotation is missing or cancelled.
    """
    annotations = task.get("annotations", [])
    if not annotations:
        return None
    ann = annotations[0]
    if ann.get("was_cancelled", False):
        return None
    result = ann.get("result", [])
    if not result:
        return None
    try:
        return result[0]["value"]["choices"][0]
    except (KeyError, IndexError):
        return None


def _split_ids(
    ids: list,
    train_ratio: float = SPLIT_TRAIN,
    val_ratio: float = SPLIT_VAL,
    seed: int = RANDOM_SEED
    ) -> tuple[set, set, set]:
    rng = np.random.default_rng(seed)
    arr = np.array(ids)
    rng.shuffle(arr)
    n = len(arr)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_ids = set(arr[:n_train].tolist())
    val_ids = set(arr[n_train:n_train + n_val].tolist())
    test_ids = set(arr[n_train + n_val:].tolist())

    print(f"  Split → train: {len(train_ids)} | "
          f"val: {len(val_ids)} | test: {len(test_ids)} images")

    return train_ids, val_ids, test_ids

## Build DataLoaders

In [19]:
def get_classification_dataloaders(
    ls_json_path:   Path  = LS_MODEL2_JSON,
    image_dir:      Path  = PROCESSED_DIR,  # cropped container images
    image_size:     int   = IMAGE_SIZE,
    batch_size:     int   = MODEL2_BATCH_SIZE,
    train_tfms:     list  = None,
    device:         str   = "cuda",
    verbose:        bool  = True
    ) -> tuple[DataLoaders, pd.DataFrame, pd.DataFrame]:
    """
    Build fastai DataLoaders for fill-level image classification (Model #2).

    Note: image_dir defaults to PROCESSED_DIR because the classification
    images should be cropped to show only the container interior
    (as specified in your Label Studio setup).

    Parameters
    ----------
    ls_json_path : Path
        Path to the JSON classification annotation file.
    image_dir : Path
        Directory containing cropped container images.
    image_size : int
        Target image size.
    batch_size : int
        Training batch size.
    train_tfms : list, optional
        Fastai transforms for training set.
    device : str
        "cuda" or "cpu".
    verbose : bool
        Print split and class information.

    Returns
    -------
    tuple[DataLoaders, pd.DataFrame, pd.DataFrame]
        (dls, train_df, test_df)
    """
    if verbose:
        print(f"Loading Label Studio annotations from: {ls_json_path}")

    tasks = _load_ls_json(ls_json_path)

    all_ids = [t["id"] for t in tasks]
    train_ids, val_ids, test_ids = _split_ids(all_ids)

    rows = []
    skipped = 0
    for task in tasks:
        task_id  = task["id"]
        label    = _extract_label(task)

        if label is None:
            print(f"  WARNING: task {task_id} has no valid annotation — skipping.")
            skipped += 1
            continue

        if label not in MODEL2_CLASSES:
            print(f"  WARNING: task {task_id} has unknown label '{label}' — skipping.")
            skipped += 1
            continue

        # Strip UUID prefix from filename: "81cc7e20-20260316_212838.jpg"
        raw_filename = task.get("file_upload", "")
        filename     = raw_filename.split("-", 1)[-1]   # → "20260316_212838.jpg"

        split = (
            "train" if task_id in train_ids else
            "valid" if task_id in val_ids   else
            "test"
        )
        rows.append({
            "image_path": str(image_dir / filename),
            "label":      label,
            "split":      split,
        })

    if skipped:
        print(f"  Skipped {skipped} task(s) with missing/invalid annotations.")

    full_df  = pd.DataFrame(rows)
    test_df  = full_df[full_df["split"] == "test"].copy()
    train_df = full_df[full_df["split"] != "test"].copy()
    train_df["is_valid"] = train_df["split"] == "valid"

    if verbose:
        print(f"  Images → train+val: {len(train_df)} | test: {len(test_df)}")
        print(f"  Label distribution (train+val):\n"
              f"{train_df['label'].value_counts().to_string()}")

    # Warn if any label in MODEL2_CLASSES is absent from the data
    found_labels = set(full_df["label"].unique())
    missing = set(MODEL2_CLASSES) - found_labels
    if missing:
        print(f"  WARNING: These classes have no images: {missing}")

    dblock = DataBlock(
        blocks     = (ImageBlock, CategoryBlock),
        get_x      = ColReader("image_path"),
        get_y      = ColReader("label"),
        splitter   = ColSplitter("is_valid"),
        item_tfms  = Resize(image_size),
        batch_tfms = train_tfms
    )

    dls = dblock.dataloaders(train_df, bs = batch_size, device = device)

    if verbose:
        print(f"  DataLoaders ready — vocab: {list(dls.vocab)}")

    return dls, train_df, test_df

In [20]:
# Building DataLoaders
dls, train_df, test_df = get_classification_dataloaders(
        ls_json_path = LS_MODEL2_JSON,
        image_dir = PROCESSED_DIR,
        image_size = IMAGE_SIZE,
        batch_size = MODEL2_BATCH_SIZE,
        train_tfms = train_tfms,
        device = DEVICE,
        verbose = True
    )

Loading Label Studio annotations from: /content/project_gn_food_estimator_v1/data/exports/model2_fill_level.json
  Loaded 100 tasks from model2_fill_level.json
  Split → train: 80 | val: 10 | test: 10 images
  Images → train+val: 90 | test: 10
  Label distribution (train+val):
label
high      19
empty     19
full      18
low       18
medium    16
  DataLoaders ready — vocab: ['empty', 'full', 'high', 'low', 'medium']


In [21]:
# Verify vocab matches config
if list(dls.vocab) != MODEL2_CLASSES:
    print(
        f"\n  WARNING: DataLoaders vocab {list(dls.vocab)} does not match "
        f"MODEL2_CLASSES {MODEL2_CLASSES}.\n"
        f"  Predictions will use DataLoaders vocab ordering. "
        f"Update MODEL2_CLASSES to match if needed."
    )

In [22]:
# Save test split
test_csv_path = LOGS_DIR / "model2_test_split.csv"
test_df.to_csv(test_csv_path, index = False)
print(f"Test split saved → {test_csv_path}")

Test split saved → /content/project_gn_food_estimator_v1/training/logs/model2_test_split.csv


In [23]:
# Sample batch sanity check
xb, yb = dls.one_batch()
print(f"Sample batch — images: {xb.shape}, labels: {yb.shape}")

Sample batch — images: torch.Size([16, 3, 512, 512]), labels: torch.Size([16])


## Class weight helper

In [24]:
def compute_class_weights(train_df: pd.DataFrame, label_col: str = "label") -> torch.Tensor:
    """
    Compute inverse-frequency class weights to handle any class imbalance.

        With 100 photos across 5 classes × 2 containers × 2 foods, some fill
    levels will inevitably have fewer examples. Passing weights to
    CrossEntropyLoss penalises mistakes on rare classes more heavily,
    preventing the model from ignoring them.

    Formula: weight[c] = total_train / (n_classes × count[c])
    A class with half the average count gets weight = 2.0.

    Returns
    -------
    torch.Tensor of shape (n_classes,) ordered to match MODEL2_CLASSES
    """
    counts = train_df[train_df["split"] == "train"][label_col].value_counts()

    # Ensure ordering matches MODEL2_CLASSES
    ordered_counts = [counts.get(cls, 1) for cls in MODEL2_CLASSES]
    total = sum(ordered_counts)
    weights = torch.tensor(
        [total / (len(MODEL2_CLASSES) * c) for c in ordered_counts],
        dtype = torch.float32
    )

    print("Class weights:")
    for cls, w in zip(MODEL2_CLASSES, weights):
        print(f"    {cls:8s} : {w:.3f}")

    return weights

In [25]:
# Computing class weights (handle minor imbalance)
class_weights = compute_class_weights(train_df)

if DEVICE == "cuda":
    class_weights = class_weights.cuda()

Class weights:
    empty    : 0.941
    full     : 0.889
    high     : 0.889
    low      : 1.067
    medium   : 1.333


# Model

Train Model #2: Fill level classifier (empty / low / medium / high / full)

**Architecture:**

EfficientNet-B0 backbone (ImageNet pre-trained) via timm, loaded through fastai's vision_learner with a classification head for 5 output classes.

**Strategy**

Two-phase training
- Phase 1 — backbone frozen, head trained for MODEL2_EPOCHS_FROZEN
- Phase 2 — full network unfrozen, fine-tuned with discriminative learning rates for MODEL2_EPOCHS_UNFROZEN

Mixed precision: fp16 enabled (safe on T4)

Important note on input images:
- Model #2 expects CROPPED images showing only the container interior.
- These should live in data/processed/ (not data/raw/).
- For training, make sure your Label Studio export references the cropped images, not the full-scene photos.

In [26]:
# Metrics
# F1Score with average="macro" is important for ordinal multi-class
# problems — it treats all classes equally regardless of frequency.
metrics = [
    accuracy,
    F1Score(average ="macro")
]

## Build learner

In [27]:
learn = vision_learner(
    dls,
    "efficientnet_b0",          # timm model name
    metrics = metrics,
    loss_func = torch.nn.CrossEntropyLoss(weight = class_weights),
    model_dir = MODELS_DIR,
    cbs = [
        SaveModelCallback(
            monitor = "valid_loss",
            fname = "model2_best",
            with_opt = True
        ),
        EarlyStoppingCallback(
            monitor = "valid_loss",
            patience = 5
        ),
        CSVLogger(fname = str(log_path))
    ]
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

## Mixed precision scaler

In [28]:
# Mixed precision
if MODEL2_MIXED_PREC and DEVICE == "cuda":
    learn = learn.to_fp16()
    print("Mixed precision (fp16) — ENABLED")

Mixed precision (fp16) — ENABLED


# Training loop

## Phase 1 - frozen backbone

In [29]:
learn.freeze()

In [30]:
# Running lr_find (Phase 1)
suggested_lr = MODEL2_LR_FROZEN

try:
    lr_finder = learn.lr_find(suggest_funcs = None, show_plot = False)
    suggested_lr = lr_finder.valley

    print(f"lr_find suggestion : {suggested_lr:.2e}  "
            f"(config default: {MODEL2_LR_FROZEN:.2e})")
except Exception as e:
    print(f"lr_find failed ({e}), using config default: {suggested_lr:.2e}")

<div></div>

lr_find failed ('NoneType' object has no attribute 'valley'), using config default: 1.00e-03


In [31]:
t0 = time.time()
learn.fit_one_cycle(
    MODEL2_EPOCHS_FROZEN,
    lr_max = suggested_lr
)

print(f"  Phase 1 complete — {time.time() - t0:.0f}s")

epoch,train_loss,valid_loss,accuracy,f1_score,time
0,2.717028,2.619473,0.300000,0.291667,00:09
1,2.515889,2.140549,0.500000,0.476190,00:04
2,2.315359,1.974270,0.400000,0.293333,00:04
3,2.157757,1.950068,0.300000,0.260000,00:05
4,2.063944,1.946345,0.300000,0.260000,00:04


Better model found at epoch 0 with valid_loss value: 2.6194732189178467.
Better model found at epoch 1 with valid_loss value: 2.1405489444732666.
Better model found at epoch 2 with valid_loss value: 1.9742695093154907.
Better model found at epoch 3 with valid_loss value: 1.9500677585601807.
Better model found at epoch 4 with valid_loss value: 1.9463447332382202.
  Phase 1 complete — 28s


## Phase 2 - full fine-tune

In [32]:
learn.unfreeze()

In [33]:
lr_unfrozen = MODEL2_LR_UNFROZEN
print(f"LR slice : {lr_unfrozen/100:.2e} → {lr_unfrozen:.2e}")

LR slice : 1.00e-06 → 1.00e-04


In [34]:
t0 = time.time()
learn.fit_one_cycle(
    MODEL2_EPOCHS_UNFROZEN,
    lr_max = slice(lr_unfrozen / 100, lr_unfrozen)
)
print(f"  Phase 2 complete — {time.time() - t0:.0f}s")

epoch,train_loss,valid_loss,accuracy,f1_score,time
0,1.637287,1.737682,0.300000,0.260000,00:08
1,1.606169,1.718234,0.400000,0.326667,00:04
2,1.415686,1.735446,0.300000,0.260000,00:04
3,1.477405,1.737147,0.400000,0.326667,00:05
4,1.484342,1.509712,0.500000,0.394286,00:04
5,1.439198,1.522079,0.500000,0.394286,00:04
6,1.451586,1.571071,0.600000,0.450000,00:05
7,1.419444,1.554878,0.600000,0.430000,00:04
8,1.426223,1.514724,0.500000,0.380952,00:04
9,1.464287,1.586332,0.500000,0.394286,00:04


Better model found at epoch 0 with valid_loss value: 1.7376816272735596.
Better model found at epoch 1 with valid_loss value: 1.7182344198226929.
Better model found at epoch 4 with valid_loss value: 1.5097123384475708.
No improvement since epoch 4: early stopping
  Phase 2 complete — 50s


# Export model

In [36]:
# Remove CSVLogger callback before export, as it can cause PicklingError
# due to holding a reference to a closed file handle.
learn.remove_cb(CSVLogger)

learn.export(MODEL2_PATH)
print(f"\n[Done] Model exported \u2192 {MODEL2_PATH}")

state_dict_path = MODELS_DIR / "model2_state_dict.pth"
torch.save(learn.model.state_dict(), state_dict_path)
print(f"[Done] State dict \u2192 {state_dict_path}")


[Done] Model exported → /content/project_gn_food_estimator_v1/models/model2_fill.pth
[Done] State dict → /content/project_gn_food_estimator_v1/models/model2_state_dict.pth


# Evaluation

In [39]:
# Ensure the learner is using the correct DataLoaders
learn.dls = dls

print(f"\n[Eval] Number of validation items: {len(dls.valid_ds)}")

if len(dls.valid_ds) > 0:
    print("\n[Eval] Validation set results:")
    val_results = learn.validate()

    if val_results is not None:
        print(f"valid_loss : {val_results[0]:.4f}")
        for i, m in enumerate(learn.metrics):
            print(f"  {m.name:15s}: {val_results[i + 1]:.4f}")
else:
    print("\n[Error] Validation dataset is empty. Check your data splitting logic.")


[Eval] Number of validation items: 10

[Eval] Validation set results:


valid_loss : 1.5097
  accuracy       : 0.5000
  f1_score       : 0.3943


In [42]:
# Confusion matrix
print("\n[Eval] Confusion matrix (validation set):")

# Explicitly use the validation dataloader from dls to avoid size mismatches
interp = ClassificationInterpretation.from_learner(learn, dl=dls.valid)
interp.print_classification_report()


[Eval] Confusion matrix (validation set):


AssertionError: ==:
50
10

## Confidence threshold check on validation set

In [43]:
print(f"Checking confidence threshold ({CONFIDENCE_THRESHOLD}):")

preds, targets = learn.get_preds(with_decoded = False)
confidences = preds.max(dim = 1).values
below_threshold = (confidences < CONFIDENCE_THRESHOLD).sum().item()
pct = below_threshold / len(confidences) * 100

print(
    f"  {below_threshold}/{len(confidences)} validation images "
    f"({pct:.1f}%) fall below the confidence threshold of "
    f"{CONFIDENCE_THRESHOLD}.\n"
    f"  These would trigger a 'retake photo' prompt in production."
)

Checking confidence threshold (0.7):


  1/10 validation images (10.0%) fall below the confidence threshold of 0.7.
  These would trigger a 'retake photo' prompt in production.


In [44]:
print(f"[Info] Training log → {log_path}")
print(f"[Info] Test split   → {test_csv_path}")

[Info] Training log → /content/project_gn_food_estimator_v1/training/logs/model2_training_log.csv
[Info] Test split   → /content/project_gn_food_estimator_v1/training/logs/model2_test_split.csv


# Quick inference check

## Visualisation